# Sprint 5 - Mixed-Domain Training (fixing catastrophic forgetting)

**Goal (see AGENT.md / final_brief_and_plan.md):** Sprint 4 fine-tuned on PlantDoc **only** and closed
most of the field-photo gap (PlantDoc F1 0.11 -> 0.51/0.56) but **forgot PlantVillage** (F1 0.95 ->
0.31) - catastrophic forgetting. This sprint trains on **both datasets in every epoch**:

1. **Variant 5 `mixed`** - warm-start from the baseline, train the head on PlantVillage + PlantDoc
   concatenated (no augmentation).
2. **Variant 6 `mixed_aug`** - same but **with** augmentation.
3. Checks **two gates**: PlantDoc F1 still beats the 0.1116 baseline, AND PlantVillage F1 stays near
   0.9501 (no forgetting).

Both datasets already share the 38-class label space (`--map-to-pv` / `class_map.json`), so
`torch.utils.data.ConcatDataset` just concatenates them - no new concepts needed.


### Quick recap of what Sprint 4 found (real run, in the CSV)
- baseline on PlantVillage test: **0.9613** acc / **0.9501** F1 (clean lab photos)
- baseline on PlantDoc test: **0.1435** acc / **0.1116** F1 (field photos) - the gap
- `pv_plus_plantdoc` on PlantDoc: **0.5391** / **0.5090** but PlantVillage F1 collapsed to **0.3116**
- `both` on PlantDoc: **0.5739** / **0.5578** but PlantVillage F1 collapsed to **0.3168**

So fine-tuning worked for field photos but wrecked the lab. Sprint 5 = both at once.


In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Requires the Sprint 0 archives (`plantvillage_raw.zip` + `plantdoc_raw.zip` + manifests) on Drive and
the Sprint 1 checkpoint `best_plantvillage_stage1.pt` (used as the warm start).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")    # durable archives (from Sprint 0)
LOCAL_RAW_DIR = Path("/content/folium_raw")             # per-session raw
LOCAL_DATA_DIR = Path("/content/folium_data")            # per-session organized splits
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as every sprint: unzip both archives from Drive into local raw, then build `train/val/test`
folders (seed 42, deterministic) plus `class_map.json`. Idempotent.


In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"
print("splits ready at", LOCAL_DATA_DIR)

## Step 4 - Train variant 5 (`mixed`)

**Mixed training** = one loader whose dataset is `ConcatDataset(PlantVillage train, PlantDoc train)`
(both in the 38-class label space), so **every epoch sees both domains**. `--mix-with plantdoc`
does this inside `ml.train`; everything else (epoch loop, best-val checkpoint) is unchanged.

Warm-start from the Sprint 1 baseline (`--init-from best_plantvillage_stage1.pt`) and train the head
only (no `--unfreeze-blocks`, no augmentation). Artifacts:
`plantvillage_mixed_epochNN.pt` + `best_plantvillage_mixed.pt`.

**Note:** PlantDoc is ~5% of each epoch (2,107 vs 43,429 images) - fine for a first test. If its
signal proves too weak we can oversample it.


In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--mix-with", "plantdoc",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--lr", "1e-3",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--tag", "mixed",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("best mixed checkpoint:", CHECKPOINT_DIR / "best_plantvillage_mixed.pt")

## Step 5 - Evaluate variant 5 on BOTH test sets

Two rows: `mixed` on `plantdoc_test` (did the gap stay closed?) and on `plantvillage_test` (did we
**keep** the lab knowledge?). Both get their own confusion matrix PNG.


In [ ]:
for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_mixed.pt"),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
        "--variant", "mixed",
    ] + extra
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    assert result.returncode == 0, f"ml.evaluate failed for {dataset}"

## Step 6 - Train variant 6 (`mixed_aug`)

Same as variant 5 **plus** augmentation (`--augment`) - the field-photo noise simulation helps the
head learn to ignore photo conditions. Artifacts: `plantvillage_mixed_aug_epochNN.pt` +
`best_plantvillage_mixed_aug.pt`.


In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--mix-with", "plantdoc",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--lr", "1e-3",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--augment",
    "--tag", "mixed_aug",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("best mixed-aug checkpoint:", CHECKPOINT_DIR / "best_plantvillage_mixed_aug.pt")

## Step 7 - Evaluate variant 6 on BOTH test sets

Same as Step 5, checkpoint `best_plantvillage_mixed_aug.pt`, variant `mixed_aug`.


In [ ]:
for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_mixed_aug.pt"),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
        "--variant", "mixed_aug",
    ] + extra
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    assert result.returncode == 0, f"ml.evaluate failed for {dataset}"

## Step 8 - The verdict: did mixed training fix the forgetting?

**Two gates** (both must pass):
1. **PlantDoc improved:** `mixed` / `mixed_aug` F1 on `plantdoc_test` beats the 0.1116 baseline.
2. **No forgetting:** PlantVillage F1 stays >= 90% of the 0.9501 baseline (>= ~0.855).

If gate 1 fails, PlantDoc's 5% share of each epoch was too weak (oversample PlantDoc next). If gate 2
fails, mixed training still lost the lab (change the recipe).


In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
cols = ["variant", "dataset", "accuracy", "precision", "recall", "f1"]
print(df[cols].to_string(index=False))

key = lambda v, d: df[(df["variant"] == v) & (df["dataset"] == d)]
base_pd = key("baseline_pv_only_no_aug", "plantdoc_test")
base_pv = key("baseline_pv_only_no_aug", "plantvillage_test")

print("\nSprint 5 - the two gates:")
for name, r in [("mixed on PlantDoc", key("mixed", "plantdoc_test")),
                ("mixed_aug on PlantDoc", key("mixed_aug", "plantdoc_test")),
                ("mixed on PlantVillage", key("mixed", "plantvillage_test")),
                ("mixed_aug on PlantVillage", key("mixed_aug", "plantvillage_test"))]:
    if not r.empty:
        row = r.iloc[0]
        print(f"  {name:26s}: accuracy {row['accuracy']:.4f} | f1 {row['f1']:.4f}")

if not base_pd.empty and not base_pv.empty:
    g1, b1 = base_pd.iloc[0]['f1'], base_pv.iloc[0]['f1']
    gate1 = [n for n in ("mixed", "mixed_aug") if not key(n, "plantdoc_test").empty
             and key(n, "plantdoc_test").iloc[0]['f1'] > g1]
    gate2 = [n for n in ("mixed", "mixed_aug") if not key(n, "plantvillage_test").empty
             and key(n, "plantvillage_test").iloc[0]['f1'] >= 0.9 * b1]
    print(f"\n  gate 1 PlantDoc F1 > {g1:.4f}:  {gate1 or 'NOT MET'}")
    print(f"  gate 2 PlantVillage F1 >= {0.9 * b1:.4f}: {gate2 or 'NOT MET'}")
    if gate1 and gate2:
        print("\nSprint 5 DONE - mixed training keeps BOTH domains: field gap closed, no forgetting.")
    else:
        print("\nSprint 5 NOT met yet - see which gate failed and why (PlantDoc oversampling / recipe).")
else:
    print("\nMissing baseline rows - run sprint4 Step 4 first (it logs baseline on plantdoc_test).")

## Step 9 - Predict a field photo and a lab photo (done-when)

Classify a couple of real PlantDoc field photos and PlantVillage lab photos with the best mixed model.
A correct PlantDoc label means the mixed model maps field photos onto the right disease; a correct
PlantVillage label means it kept the lab knowledge.


In [ ]:
for folder in (LOCAL_DATA_DIR / "plantdoc" / "test", LOCAL_DATA_DIR / "plantvillage" / "test"):
    images = sorted(folder.glob("*/*.jpg"))[:2]
    for image in images:
        cmd = [
            sys.executable, "-m", "ml.predict",
            "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_mixed_aug.pt"),
            "--image", str(image),
            "--topk", "3",
        ]
        subprocess.run(cmd, cwd=str(REPO_DIR))
        print("  (true class folder:", image.parent.name, ")")

## Where things live

**On Google Drive (durable):**
```
folium/checkpoints/best_plantvillage_mixed.pt      variant 5 (mixed)
folium/checkpoints/best_plantvillage_mixed_aug.pt  variant 6 (mixed + augmentation)
folium/results/ablation_results.csv                all rows (the paper's source of truth)
folium/results/cm_mixed.png / cm_mixed_aug.png     PlantDoc + PlantVillage matrices
```
Every number in the CSV comes from an actual logged run - nothing fabricated.
